<a href="https://colab.research.google.com/github/hjiwoong/DL/blob/main/day08_practice3_%EC%A6%9D%EA%B0%95_%EC%8B%A4%EC%B8%A1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 증강 효과 실측
# "데이터가 적을 때 증강이 도움이 된다"를 W&B로 기록하여 실측 - 그런데 결과가 데이터에 따라 갈린다


In [2]:
!pip install -U wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.1/26.1 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 19.1 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.63b1
    Uninstalling opentelemetry-semantic-conventions-0.63b1:
      Successfully uninstalled opentelemetry-semantic-conventions-0.63b1
  Attempting uninstall: opentelemetry-sdk
    Found existing installation: opentelemetry-sdk 1.42.1
    Uninstalling opentelemetry-sdk-1.

In [3]:
import os

os.environ["WANDB_MODE"] = "online"

import wandb

print("WANDB_MODE =", os.environ.get("WANDB_MODE"))

WANDB_MODE = online


In [4]:
# 로그인
import os
from google.colab import userdata
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

import wandb
wandb.login()

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: gimm00999 (gimm00999-kwu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("장치:", device)

DATASET = "auto"
N_TRAIN = 5000
EPOCHS = 12

장치: cuda


In [6]:
# [셀 1] 데이터 준비 - 증강 유/무 두 벌의 학습셋
def get_datasets(root="./data"):
  if DATASET in ("auto", "cifar10"):
    aug32 = [transforms.RandomHorizontalFlip(), transforms.RandomRotation(32)]
    tr_aug = datasets.CIFAR10(root, train=True, download=True, transform=transforms.Compose(aug32 + [transforms.ToTensor()])) # 증강 학습셋
    tr_no = datasets.CIFAR10(root, train=True, download=True, transform=transforms.ToTensor()) # 무증강 학습셋
    te = datasets.CIFAR10(root, train=False, download=True, transform=transforms.ToTensor()) # 테스트셋(증강 없음)
    return "CIFAR-10", tr_aug, tr_no, te

name, train_aug, train_no, test_set = get_datasets()
C = train_no[0][0].shape[0]
print(f"데이터: {name} | 학습 제한 {N_TRAIN}장 (원래{len(train_no)}장)")

# 같은 5000장을 두 실험이 공유 -> 공정 비교 (증강 유무만 차이나게)
idx = torch.randperm(len(train_no), generator=torch.Generator().manual_seed(0))[:N_TRAIN].tolist()
test_loader = DataLoader(test_set, batch_size=512, shuffle=False) # 테스트 로더(고정)

100%|██████████| 170M/170M [48:42<00:00, 58.3kB/s]


데이터: CIFAR-10 | 학습 제한 5000장 (원래50000장)


In [7]:
# [셀 2] 공동 모델, 평가 도구
def make_cnn(in_ch, img_size):
  feat = img_size // 4
  return nn.Sequential(
      nn.Conv2d(in_ch, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
      nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
      nn.Flatten(), nn.Linear(64 * feat * feat, 256), nn.ReLU(),
      nn.Linear(256, 10)
  )
def accuracy(model, loader):
  model.eval()
  c=t=0
  with torch.no_grad():
    for x, y in loader:
      x, y=x.to(device), y.to(device)
      c += (model(x).argmax(1)==y).sum().item(); t += len(y)
    return c/t # 정확도 (맞은 수/전체 수)

In [8]:
# [셀 3] 두 run - 증강 없음 vs 증강 있음 (W&B 기록)
results = {}
for use_aug in [False, True]:
  run_name = "with_aug" if use_aug else "no_aug"
  run = wandb.init(project="dl-day08-augmentation", name=run_name,
                   config={"aug": use_aug, "n_train": N_TRAIN, "epochs": EPOCHS, "dataset":name, "seed":42},
                   reinit = "finish_previous")
  dataset = train_aug if use_aug else train_no
  train_loader = DataLoader(Subset(dataset, idx), batch_size=128, shuffle=True, generator=torch.Generator().manual_seed(42))
  img_size = dataset[0][0].shape[-1]

  torch.manual_seed(42)
  model = make_cnn(C, img_size).to(device)
  loss_fn = nn.CrossEntropyLoss()
  opt = torch.optim.Adam(model.parameters(), lr=0.001)

  for epoch in range(EPOCHS):
    model.train()
    for x, y in train_loader:
      x, y = x.to(device), y.to(device)
      loss = loss_fn(model(x), y)
      opt.zero_grad(); loss.backward(); opt.step()
    te_acc = accuracy(model, test_loader)
    wandb.log({"test_acc": te_acc})

  tr_acc = accuracy(model, train_loader)
  results[run_name] = (tr_acc, te_acc)
  wandb.summary["final_train_acc"] = tr_acc
  wandb.summary["final_test_acc"] = te_acc
  wandb.finish
  print(f"[{run_name:8s}] train {tr_acc:.3f} | test{te_acc:.3f} | 격차 {tr_acc - te_acc:+.3f}")

[no_aug  ] train 0.660 | test0.539 | 격차 +0.121


test_acc,▁▃▃▅▅▆▇▆▇█▆█
final_test_acc,0.5387
final_train_acc,0.66
test_acc,0.5387


[with_aug] train 0.519 | test0.515 | 격차 +0.004
